# GMW v4 Extent Vector Tiles Pipeline

Processes all yearly geopackage files from the GMW v4 extent dataset and publishes them as Mapbox tilesets.

Steps for each year file in `data/GMW_extent_v4/`:
1. Decompress `.gpkg.gz` → `.gpkg`
2. Convert to GeoJSON using **mapshaper** (with geometry clean + rewind)
3. Generate `.mbtiles` using **tippecanoe** (source layer named `gmw_v4_extent_{year}`)
4. Upload to Mapbox as tileset `{MAPBOX_USERNAME}.gmw-v4-extent-{year}`

**Prerequisites:**
- `mapshaper` available on `$PATH` (`npm install -g mapshaper`)
- `tippecanoe` available on `$PATH`
- `boto3` Python package (`pip install boto3`)
- Environment variables set: `MAPBOX_USERNAME`, `MAPBOX_ACCESS_TOKEN`

In [1]:
import dotenv
import gc
import geopandas as gpd
import gzip
import logging
import os
import re
import shutil
import subprocess
from pathlib import Path

import boto3
import requests

In [2]:
# Paths — notebook lives at notebooks/Lab/data_processing/layers/,
# so 4 parents up reaches the repo data/ root.
WORK_DIR = Path(os.getcwd())
DATA_DIR = WORK_DIR.parents[3] / "data"

INPUT_DIR = DATA_DIR / "GMW_extent_v4"
OUTPUT_DIR = DATA_DIR / "processed" / "GMW_extent_v4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert INPUT_DIR.exists(), f"Input directory not found: {INPUT_DIR}"

# Mapbox settings — set these in your .env or export before running.
# MAPBOX_ACCESS_TOKEN must be a SECRET token (sk.*) with uploads:write scope.
# Public tokens (pk.*) will return 404 from the Uploads API.
dotenv.load_dotenv()
MAPBOX_USERNAME = os.environ.get("MAPBOX_USER", "")
MAPBOX_ACCESS_TOKEN = os.environ.get("MAPBOX_VECTOR_TOKEN_2", "")

assert MAPBOX_USERNAME, "MAPBOX_USER environment variable is not set"
assert MAPBOX_ACCESS_TOKEN, "MAPBOX_ACCESS_TOKEN environment variable is not set"
assert MAPBOX_ACCESS_TOKEN.startswith("sk."), (
    f"MAPBOX_ACCESS_TOKEN must be a secret token (sk.*) with uploads:write scope. "
    f"Got a token starting with '{MAPBOX_ACCESS_TOKEN[:3]}'. "
    "Create one at https://account.mapbox.com/access-tokens/"
)

TILESET_PREFIX = "gmw-v4-extent"
SOURCE_LAYER_PREFIX = "gmw_v4_extent"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
print(f"Input : {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Mapbox username: {MAPBOX_USERNAME}")
print(f"Token type: secret ✓")

Input : /Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/GMW_extent_v4
Output: /Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/processed/GMW_extent_v4
Mapbox username: globalmangrovewatch
Token type: secret ✓


## Helper functions

In [6]:
def extract_year(filename: str) -> str | None:
    """Parse the 4-digit year from a GMW v4 filename like gmw_v4112_1985_mng_ext_vec.gpkg.gz."""
    match = re.search(r"gmw_v4\d+_(\d{4})_", filename)
    return match.group(1) if match else None


def decompress_gpkg(source_gz: Path, dest_dir: Path) -> Path:
    """Decompress a .gpkg.gz file into dest_dir. Skips if already done."""
    dest = dest_dir / source_gz.stem  # strips .gz, keeps .gpkg
    if not dest.exists():
        logging.info(f"Decompressing {source_gz.name}...")
        with gzip.open(source_gz, "rb") as f_in, open(dest, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    return dest


def convert_to_geojson(gpkg_path: Path, year: str, update: bool = False) -> Path:
    """Convert a .gpkg to a cleaned GeoJSON.

    Reads with geopandas (GDAL handles encoding), drops all non-geometry
    columns to minimise the in-memory footprint, writes a UTF-8 GeoJSON,
    then immediately frees the GeoDataFrame and runs gc.collect() before
    handing off to mapshaper for geometry cleanup.
    Removes the source .gpkg and the tmp GeoJSON when no longer needed.
    """
    out_path = gpkg_path.parent / f"{SOURCE_LAYER_PREFIX}_{year}.json"
    if update or not out_path.exists():
        tmp_path = out_path.with_suffix(".tmp.json")

        # Step 1: read → keep geometry only → write UTF-8 GeoJSON → free memory
        logging.info(f"Reading {gpkg_path.name} with geopandas...")
        gdf = gpd.read_file(gpkg_path, driver="GPKG")
        gdf = gdf[["geometry"]]
        gdf.to_file(tmp_path, driver="GeoJSON")
        del gdf
        gc.collect()

        # Remove the decompressed gpkg immediately — it's no longer needed
        gpkg_path.unlink(missing_ok=True)

        # Step 2: mapshaper cleans geometry; heap raised to 8 GB
        logging.info("Cleaning geometry with mapshaper...")
        env = os.environ.copy()
        env["NODE_OPTIONS"] = "--max-old-space-size=8192"
        subprocess.run(
            f"mapshaper {tmp_path} -clean allow-overlaps rewind -o format=geojson {out_path} force",
            shell=True,
            check=True,
            env=env,
        )
        tmp_path.unlink(missing_ok=True)
    return out_path


def create_mbtiles(geojson_path: Path, year: str, update: bool = False) -> Path:
    """Generate an .mbtiles file from a GeoJSON using tippecanoe.

    The source layer inside the tileset is named gmw_v4_extent_{year}.
    Deletes the intermediate GeoJSON after a successful run.
    """
    out_path = geojson_path.parent / f"{SOURCE_LAYER_PREFIX}_{year}.mbtiles"
    if update or not out_path.exists():
        layer_name = f"{SOURCE_LAYER_PREFIX}_{year}"
        logging.info(f"Creating mbtiles for year {year} (layer: {layer_name})...")
        cmd = (
            f"tippecanoe -zg -f -P "
            f"-l {layer_name} "
            f"--extend-zooms-if-still-dropping "
            f"-q "
            f"-o {out_path} {geojson_path}"
        )
        subprocess.run(cmd, shell=True, check=True)
        geojson_path.unlink(missing_ok=True)
    return out_path


def upload_to_mapbox(mbtiles_path: Path, year: str) -> dict:
    """Upload an .mbtiles file to Mapbox via the Uploads API.

    Returns the upload record dict from the Mapbox API.
    Tileset ID: {MAPBOX_USERNAME}.gmw-v4-extent-{year}
    """
    full_tileset = f"{MAPBOX_USERNAME}.{TILESET_PREFIX}-{year}"

    # 1. Retrieve temporary S3 credentials from Mapbox
    logging.info(f"Fetching Mapbox upload credentials for {full_tileset}...")
    creds_resp = requests.post(
        f"https://api.mapbox.com/uploads/v1/{MAPBOX_USERNAME}/credentials",
        params={"access_token": MAPBOX_ACCESS_TOKEN},
    )
    creds_resp.raise_for_status()
    creds = creds_resp.json()

    # 2. Upload the mbtiles file to the Mapbox-provisioned S3 bucket
    logging.info(f"Uploading {mbtiles_path.name} to S3...")
    s3 = boto3.client(
        "s3",
        aws_access_key_id=creds["accessKeyId"],
        aws_secret_access_key=creds["secretAccessKey"],
        aws_session_token=creds["sessionToken"],
        region_name="us-east-1",
    )
    s3.upload_file(str(mbtiles_path), creds["bucket"], creds["key"])

    # 3. Register the tileset with Mapbox
    logging.info(f"Registering tileset {full_tileset}...")
    upload_resp = requests.post(
        f"https://api.mapbox.com/uploads/v1/{MAPBOX_USERNAME}",
        params={"access_token": MAPBOX_ACCESS_TOKEN},
        json={
            "url": creds["url"],
            "tileset": full_tileset,
            "name": f"GMW v4 Extent {year}",
        },
    )
    upload_resp.raise_for_status()
    return upload_resp.json()


def check_upload_status(upload_id: str) -> dict:
    """Poll the status of a Mapbox upload by its upload ID."""
    resp = requests.get(
        f"https://api.mapbox.com/uploads/v1/{MAPBOX_USERNAME}/{upload_id}",
        params={"access_token": MAPBOX_ACCESS_TOKEN},
    )
    resp.raise_for_status()
    return resp.json()

## Pipeline

In [4]:
# Validate decompressed .gpkg files — removes any that can't be opened by GDAL
# Run this once to clean up leftovers from failed runs, then re-run the pipeline.

for gpkg in sorted(OUTPUT_DIR.glob("*.gpkg")):
    result = subprocess.run(
        ["ogrinfo", "-so", "-al", str(gpkg)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"CORRUPT — removing: {gpkg.name}")
        gpkg.unlink()
    else:
        print(f"OK: {gpkg.name}")

In [5]:
# List and verify input files
input_files = sorted(INPUT_DIR.glob("*.gpkg.gz"))
print(f"Found {len(input_files)} files:\n")
for f in input_files:
    year = extract_year(f.name)
    print(f"  {f.name}  →  year: {year}")

Found 41 files:

  gmw_v4112_1985_mng_ext_vec.gpkg.gz  →  year: 1985
  gmw_v4112_1986_mng_ext_vec.gpkg.gz  →  year: 1986
  gmw_v4112_1987_mng_ext_vec.gpkg.gz  →  year: 1987
  gmw_v4112_1988_mng_ext_vec.gpkg.gz  →  year: 1988
  gmw_v4112_1989_mng_ext_vec.gpkg.gz  →  year: 1989
  gmw_v4112_1990_mng_ext_vec.gpkg.gz  →  year: 1990
  gmw_v4112_1991_mng_ext_vec.gpkg.gz  →  year: 1991
  gmw_v4112_1992_mng_ext_vec.gpkg.gz  →  year: 1992
  gmw_v4112_1993_mng_ext_vec.gpkg.gz  →  year: 1993
  gmw_v4112_1994_mng_ext_vec.gpkg.gz  →  year: 1994
  gmw_v4112_1995_mng_ext_vec.gpkg.gz  →  year: 1995
  gmw_v4112_1996_mng_ext_vec.gpkg.gz  →  year: 1996
  gmw_v4112_1997_mng_ext_vec.gpkg.gz  →  year: 1997
  gmw_v4112_1998_mng_ext_vec.gpkg.gz  →  year: 1998
  gmw_v4112_1999_mng_ext_vec.gpkg.gz  →  year: 1999
  gmw_v4112_2000_mng_ext_vec.gpkg.gz  →  year: 2000
  gmw_v4112_2001_mng_ext_vec.gpkg.gz  →  year: 2001
  gmw_v4112_2002_mng_ext_vec.gpkg.gz  →  year: 2002
  gmw_v4112_2003_mng_ext_vec.gpkg.gz  →  year: 

In [7]:
# Process all files
upload_results = {}

input_files_test = input_files[:1]  # For testing, process only the first file. Remove or adjust for full run.

for gz_file in input_files:
    year = extract_year(gz_file.name)
    if not year:
        logging.warning(f"Could not extract year from {gz_file.name} — skipping.")
        continue

    logging.info(f"\n{'=' * 60}")
    logging.info(f"Processing year {year} ({gz_file.name})")

    # 1. Decompress
    gpkg_file = decompress_gpkg(gz_file, OUTPUT_DIR)

    # 2. Convert to GeoJSON with mapshaper
    geojson_file = convert_to_geojson(gpkg_file, year, update=True)

    # 3. Create mbtiles with tippecanoe (deletes geojson on completion)
    mbtiles_file = create_mbtiles(geojson_file, year, update=True)

    # 4. Upload to Mapbox
    result = upload_to_mapbox(mbtiles_file, year)
    upload_results[year] = result
    logging.info(f"Upload queued — id: {result.get('id')}, tileset: {result.get('tileset')}")

    # 5. Remove intermediate decompressed gpkg
    gpkg_file.unlink(missing_ok=True)

logging.info("\nAll files processed.")
print(f"\nUpload IDs:")
for year, r in upload_results.items():
    print(f"  {year}: {r.get('id')} — {r.get('tileset')}")

INFO: 
INFO: Processing year 1985 (gmw_v4112_1985_mng_ext_vec.gpkg.gz)
INFO: Decompressing gmw_v4112_1985_mng_ext_vec.gpkg.gz...
INFO: Reading gmw_v4112_1985_mng_ext_vec.gpkg with geopandas...
/Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/.venv/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(
INFO: Created 1,156,657 records
INFO: Cleaning geometry with mapshaper...
[clean] Retained 1,156,657 of 1,156,657 features
[o] Wrote /Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/processed/GMW_extent_v4/gmw_v4_extent_1985.json
INFO: Creating mbtiles for year 1985 (layer: gmw_v4_extent_1985)...
/Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/processed/GMW_extent_v4/gmw_v4_extent_1985.json:51: Warning: not finding any GeoJSON features or geometries in input yet after 50 objects.
/Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/processed/GMW_extent_v4/gmw_v4_extent_1985


Upload IDs:
  1985: cmp3y6nzu0n1m1nn46097zlyy — globalmangrovewatch.gmw-v4-extent-1985
  1986: cmp3ydfzx0nhx1op3uxz9w21p — globalmangrovewatch.gmw-v4-extent-1986
  1987: cmp3yk1q90n5e1nn498c21bac — globalmangrovewatch.gmw-v4-extent-1987
  1988: cmp3yqqif071z1mo14mxbjeja — globalmangrovewatch.gmw-v4-extent-1988
  1989: cmp3yxeo10vnp1oqpxceb57p9 — globalmangrovewatch.gmw-v4-extent-1989
  1990: cmp3z473s00h81no11zlda7k1 — globalmangrovewatch.gmw-v4-extent-1990
  1991: cmp3zatb005s621pewegvidpm — globalmangrovewatch.gmw-v4-extent-1991
  1992: cmp3zhpu90ngb1qp3wjf9jfz5 — globalmangrovewatch.gmw-v4-extent-1992
  1993: cmp3zohbz07jt1po1f29cmlsb — globalmangrovewatch.gmw-v4-extent-1993
  1994: cmp3zv8yd00rb1nqh4x01emzn — globalmangrovewatch.gmw-v4-extent-1994
  1995: cmp401py20nhf1on4vbkxcg60 — globalmangrovewatch.gmw-v4-extent-1995
  1996: cmp408cbi0nyr1op3vxplqg56 — globalmangrovewatch.gmw-v4-extent-1996
  1997: cmp40ewz70c1w1oqleg1xq8gp — globalmangrovewatch.gmw-v4-extent-1997
  1998: cmp4

## Check upload status

Mapbox processes uploads asynchronously. Use the cell below to poll status for a specific upload ID returned above.

In [8]:
# Check status for all uploads
for year, result in upload_results.items():
    upload_id = result.get("id")
    if upload_id:
        status = check_upload_status(upload_id)
        print(f"{year}: {status.get('complete')} — {status.get('error') or 'ok'}")

1985: False — ok
